# L335 MiniProject: baseline setup for Task 2
Sets up both baselines with the versions from their READMEs (PGD: PyTorch 2.3.1, P2P-Bridge: PyTorch 2.1.2, both with CUDA 11.8), the PU-Net test data from ScoreDenoise and both checkpoints.

**How to use:** 
- Runtime, Run all (T4 GPU).
- Step 1b asks for access to Google Drive; confirm it with the account that owns this notebook (the cell waits up to 10 min).
- With the cache in `DRIVE_ROOT/cache` a new session takes about 17 min after the Drive confirmation (tested on 2026-11-14), most of it Step 3b.
- Without the cache (first run, or another GPU type) the CUDA extensions are compiled (about 45 min), and Step 5 then saves the cache.
- Before running P2P-Bridge: `source /content/p2pb_env.sh` (the compiler settings of the cached pvcnn build).
- Everything under `/content` is deleted when the session ends; save results to `DRIVE_ROOT/results`.

In [ ]:
# Settings: the only cell you may need to edit
# DRIVE_ROOT is a folder in your own Google Drive; Step 1b creates it, Step 5 fills cache/ (first run ~45 min, later ~17 min)
DRIVE_ROOT = "/content/drive/MyDrive/L335_MiniProject"
# code versions used for Task 2 (same commits as the submodules in external/)
PGD_COMMIT = "b5dc8be"
P2PB_COMMIT = "eb26f07"
import os
os.environ.update(L335_DRIVE=DRIVE_ROOT, PGD_COMMIT=PGD_COMMIT, P2PB_COMMIT=P2PB_COMMIT)  # read by the %%bash cells
print("Drive folder:", DRIVE_ROOT)


In [1]:
# Step 0: check that a GPU is attached
!nvidia-smi

Sun Nov 15 08:40:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Step 1: clone the two baselines (public repos, no token needed) and pin the commits from the Settings cell
%cd /content
!git clone -q https://github.com/git-guocc/PGD.git && git -C PGD checkout -q $PGD_COMMIT
!git clone -q https://github.com/matvogel/P2P-Bridge.git && git -C P2P-Bridge checkout -q $P2PB_COMMIT
!git -C PGD log -1 --oneline
!git -C P2P-Bridge log -1 --oneline
!ls PGD P2P-Bridge

/content
Cloning into 'PGD'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 75 (delta 15), reused 74 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 9.84 MiB | 13.26 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Cloning into 'P2P-Bridge'...
remote: Enumerating objects: 534, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 534 (delta 9), reused 9 (delta 9), pack-reused 515 (from 1)
Receiving objects: 100% (534/534), 36.23 MiB | 15.27 MiB/s, done.
Resolving deltas: 100% (82/82), done.
b5dc8be (HEAD -> main, origin/main, origin/HEAD) Initial commit with project setup and basic structure.
eb26f07 (HEAD -> main, origin/main, origin/HEAD) Adding BibTeX
P2P-Bridge:
assets		   evaluate_objects.py	requirements_full.txt  third_party
configs		   evaluate_rooms.py	requirements.txt       train.py
data		   in

In [4]:
# Step 1b: Google Drive holds the cache (compiled extensions, data, checkpoints) and the results
# a dialog asks for access to Google Drive; confirm it with the account that owns this notebook (waits up to 10 min)
from google.colab import drive
drive.mount('/content/drive', timeout_ms=600000)
import os
os.makedirs(DRIVE_ROOT + '/cache', exist_ok=True)
os.makedirs(DRIVE_ROOT + '/results', exist_ok=True)
print('cache:', sorted(os.listdir(DRIVE_ROOT + '/cache')))

Mounted at /content/drive
cache: ['PVDS_PUNet', 'data.zip', 'p2pb-explicit.txt', 'pgd-explicit.txt', 'torch_extensions.tar.gz', 'wheels']


In [5]:
# Step 2: versions of the preinstalled software
import sys, torch, numpy
print("Python", sys.version.split()[0])
print("PyTorch", torch.__version__, "built with CUDA", torch.version.cuda)
print("GPU", torch.cuda.get_device_name(0))
print("NumPy", numpy.__version__)
!nvcc --version | tail -1

Python 3.13.16
PyTorch 2.14.1+cu130 built with CUDA 13.0
GPU Tesla T4
NumPy 2.4.6
Build cuda_13.0.r13.0/compiler.36424714_0


In [6]:
%%bash
# Step 3a: micromamba creates isolated environments with the original versions
cd /content
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
/content/bin/micromamba --version
gcc --version | head -1
lsb_release -ds
df -h /content | tail -1

2.10.0
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Ubuntu 24.04.4 LTS
overlay         236G  203M  236G   1% /


In [7]:
%%bash
# Step 3b: both environments with the versions from the READMEs, created one after the other
# PGD: Python 3.10, PyTorch 2.3.1, CUDA 11.8; P2P-Bridge: Python 3.10, PyTorch 2.1.2, CUDA 11.8
# with the exact package lists from the Drive cache there is no dependency solving (much faster);
# without them the specs below are solved, and the resulting lists are saved to the cache for next time
# gcc 11 is included because nvcc 11.8 refuses the system gcc 13; existing environments are skipped
export MAMBA_ROOT_PREFIX=/content/mamba
M=/content/bin/micromamba
C=$L335_DRIVE/cache
CH="-c pytorch3d -c pytorch -c nvidia/label/cuda-11.8.0 -c nvidia -c conda-forge"
TOOLS="pytorch-cuda=11.8 pytorch3d cuda-nvcc cuda-cudart-dev cuda-libraries-dev gcc_linux-64=11 gxx_linux-64=11 mkl<2024.1"
if test ! -x /content/mamba/envs/pgd/bin/python; then
if test -f $C/pgd-explicit.txt; then $M create -y -n pgd --file $C/pgd-explicit.txt > /content/pgd_env.log 2>&1;
else $M create -y -n pgd $CH python=3.10 pytorch=2.3.1 torchvision=0.18.1 $TOOLS "numpy<2" > /content/pgd_env.log 2>&1; fi;
fi;
if test ! -x /content/mamba/envs/p2pb/bin/python; then
if test -f $C/p2pb-explicit.txt; then $M create -y -n p2pb --file $C/p2pb-explicit.txt > /content/p2pb_env.log 2>&1;
else $M create -y -n p2pb $CH python=3.10 pytorch=2.1.2 torchvision=0.16.2 $TOOLS numpy=1.26 > /content/p2pb_env.log 2>&1; fi;
fi;
for e in pgd p2pb; do test -f $C/$e-explicit.txt || $M env export -n $e --explicit --md5 > $C/$e-explicit.txt; done
for e in pgd p2pb; do /content/mamba/envs/$e/bin/python -c "import torch, pytorch3d; print('$e', torch.__version__, torch.version.cuda, pytorch3d.__version__, torch.cuda.is_available())"; done

pgd 2.3.1 11.8 0.7.8 True
p2pb 2.1.2 11.8 0.7.8 True


In [8]:
%%bash
# Step 3c: pip packages from the PGD README; its two CUDA extensions come from the Drive cache,
# or are compiled (about 11 min) if there is no cache yet
ENV=/content/mamba/envs/pgd
PY=$ENV/bin/python
W=$L335_DRIVE/cache/wheels/pgd
$PY -m pip install -q "numpy<2" lightning==2.4.0 pytorch-lightning==2.4.0 iopath==0.1.10 torch-geometric==2.6.1 point-cloud-utils==0.31.0 pandas tensorboard tqdm scipy
$PY -m pip install -q "numpy<2" pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.3.1+cu118.html
if ls $W/*.whl > /dev/null 2>&1; then
$PY -m pip install -q --no-deps $W/*.whl && echo "chamfer3d and pointops installed from the Drive cache"
else
# compile with CUDA 11.8 and gcc 11 from the environment, only for the T4 (compute capability 7.5)
export PATH=$ENV/bin:$PATH CUDA_HOME=$ENV TORCH_CUDA_ARCH_LIST=7.5 MAX_JOBS=2
export CC=$ENV/bin/x86_64-conda-linux-gnu-gcc CXX=$ENV/bin/x86_64-conda-linux-gnu-g++
cd /content/PGD/chamfer3d && $PY setup.py install > /content/pgd_chamfer3d.log 2>&1; echo "chamfer3d exit $?"
cd /content/PGD/pointops && $PY setup.py install > /content/pgd_pointops.log 2>&1; echo "pointops exit $?"
fi

chamfer3d and pointops installed from the Drive cache


In [9]:
%%bash
# Step 3d: pip packages of P2P-Bridge; its 7 CUDA extensions and pvcnn come from the Drive cache,
# or are compiled with its install.sh (about 20 min) if there is no cache yet
# /content/p2pb_env.sh holds the compiler settings: source it before running P2P-Bridge, otherwise
# pvcnn (compiled on first import) does not match the cached build and is compiled again
cat > /content/p2pb_env.sh <<'EOF'
E=/content/mamba/envs/p2pb
export PATH=$E/bin:$PATH CUDA_HOME=$E CONDA_PREFIX=$E TORCH_CUDA_ARCH_LIST=7.5 MAX_JOBS=2
export CC=$E/bin/x86_64-conda-linux-gnu-gcc CXX=$E/bin/x86_64-conda-linux-gnu-g++
EOF
source /content/p2pb_env.sh;
C=$L335_DRIVE/cache
cd /content/P2P-Bridge
python -m pip install -q torch_cluster -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
# setuptools 70+ no longer ships pkg_resources, which PyTorch 2.1 needs to build extensions
python -m pip install -q "setuptools<70"
# openpoints imports these, but they are missing from P2P-Bridge's requirements.txt
python -m pip install -q termcolor tabulate;
if ls $C/wheels/p2pb/*.whl > /dev/null 2>&1; then
python -m pip install -q -r requirements.txt
python -m pip install -q --no-deps $C/wheels/p2pb/*.whl && echo "7 extensions installed from the Drive cache"
# restore the pvcnn build; the fresh clone makes its sources newer than that build, so their dates
# are set back (content unchanged), otherwise ninja would compile pvcnn again
tar -xzf $C/torch_extensions.tar.gz -C /root/.cache && find third_party/pvcnn -type f -exec touch -d 2001-01-01 {} + && echo "pvcnn restored from the Drive cache"
else
bash install.sh > /content/p2pb_install.log 2>&1; echo "install.sh exit $?"
fi

7 extensions installed from the Drive cache
pvcnn restored from the Drive cache


In [10]:
%%bash
# Step 3e: quick check that both baselines import in their own environments, including pvcnn
# (loads in a few seconds from the cache; on the very first run it is compiled here, about 5 min)
cd /content/PGD && /content/mamba/envs/pgd/bin/python -c "from models.pgd import PGDModel; import chamfer3d, pointops_cuda; print('PGD ok')"
source /content/p2pb_env.sh;
cd /content/P2P-Bridge && python -c "from models.model_loader import load_diffusion; from models.evaluation import Evaluator; import third_party.pvcnn.functional.backend; print('P2P-Bridge ok')"

PGD ok
ninja: no work to do.
P2P-Bridge ok


Using /root/.cache/torch_extensions/py310_cu118 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py310_cu118/_pvcnn_backend/build.ninja...
Building extension module _pvcnn_backend...
Using envvar MAX_JOBS (2) as the number of workers...
Loading extension module _pvcnn_backend...


In [11]:
%%bash
# Step 4: test data (PU-Net and PC-Net from ScoreDenoise) and the P2P-Bridge checkpoint;
# taken from the Drive cache if possible, otherwise downloaded; skips what is already there
C=$L335_DRIVE/cache
cd /content
test -f data.zip || cp $C/data.zip . || gdown 1LdrQhcqDJa-A5Ngiywt0C_KFD60vRqfA -O data.zip
test -d data/examples || unzip -q -o data.zip -d data
mkdir -p P2P-Bridge/pretrained
test -f P2P-Bridge/pretrained/PVDS_PUNet/latest.pth || cp -r $C/PVDS_PUNet P2P-Bridge/pretrained/ || gdown --folder 1w_uhc2XeNqmqhsh4FArg5Pfx9EG1ryqg -O P2P-Bridge/pretrained/PVDS_PUNet
# both repos read the same files: PGD from ./data, P2P-Bridge from ./data/objects
ln -sfn /content/data PGD/data
ln -sfn /content/data P2P-Bridge/data/objects
echo "noisy inputs:"; ls PGD/data/examples | grep PUNet
echo "PU-Net test shapes: $(ls data/PUNet/meshes/test | wc -l) meshes, $(ls data/examples/PUNet_10000_poisson_0.01 | wc -l) noisy clouds per setting"
ls -la PGD/pretrained P2P-Bridge/pretrained/PVDS_PUNet | grep -E 'ckpt|pth|yaml'

noisy inputs:
PUNet_10000_poisson_0.01
PUNet_10000_poisson_0.02
PUNet_10000_poisson_0.03
PUNet_50000_poisson_0.01
PUNet_50000_poisson_0.02
PUNet_50000_poisson_0.03
PU-Net test shapes: 20 meshes, 20 noisy clouds per setting
-rw------- 1 root root 423653217 Nov 15 09:05 latest.pth
-rw------- 1 root root      1403 Nov 15 09:05 opt.yaml
-rw-r--r-- 1 root root 11838468 Nov 15 08:40 PGD.ckpt


In [ ]:
%%bash
# Step 5 (once per GPU type, or after changing the CUDA code of a baseline): pack the compiled extensions
# as wheels, compile pvcnn, and save them together with data and checkpoint to Google Drive
C=$L335_DRIVE/cache
if ls $C/wheels/p2pb/*.whl > /dev/null 2>&1; then echo "cache already in Drive, nothing to do"; ls $C; exit 0; fi
OUT=/content/cache_local
mkdir -p $OUT/wheels/pgd $OUT/wheels/p2pb $C
E=/content/mamba/envs/pgd
export PATH=$E/bin:/usr/local/bin:/usr/bin:/bin CUDA_HOME=$E TORCH_CUDA_ARCH_LIST=7.5 MAX_JOBS=2 CC=$E/bin/x86_64-conda-linux-gnu-gcc CXX=$E/bin/x86_64-conda-linux-gnu-g++
for d in chamfer3d pointops; do cd /content/PGD/$d && python setup.py bdist_wheel -d $OUT/wheels/pgd >> $OUT/wheels.log 2>&1; echo "PGD $d exit $?"; done
# P2P-Bridge with exactly the settings from /content/p2pb_env.sh, so the saved pvcnn build matches later sessions
source /content/p2pb_env.sh;
# with setuptools<70, bdist_wheel needs an older wheel package
python -m pip install -q "wheel<0.44"
for d in metrics/chamfer3D metrics/emd_assignment metrics/PyTorchEMD third_party/openpoints/cpp/pointops third_party/openpoints/cpp/pointnet2_batch third_party/openpoints/cpp/emd third_party/openpoints/cpp/chamfer_dist; do cd /content/P2P-Bridge/$d && python setup.py bdist_wheel -d $OUT/wheels/p2pb >> $OUT/wheels.log 2>&1; echo "P2P-Bridge $d exit $?"; done
cd /content/P2P-Bridge && python -c "import third_party.pvcnn.functional.backend" >> $OUT/wheels.log 2>&1; echo "pvcnn exit $?"
tar -czf $OUT/torch_extensions.tar.gz -C /root/.cache torch_extensions
cp -r $OUT/wheels $OUT/torch_extensions.tar.gz /content/data.zip $C/
cp -r /content/P2P-Bridge/pretrained/PVDS_PUNet $C/
ls $C $C/wheels/*; du -sh $C

cache already in Drive, nothing to do
data.zip
p2pb-explicit.txt
pgd-explicit.txt
PVDS_PUNet
torch_extensions.tar.gz
wheels


In [ ]:
%%bash
# Task 2, step 1: PGD with its documented settings 

# 2.5% (--niters 3) is not run: the ScoreDenoise data has no noisy inputs at 2.5%
date +%T
cd /content/PGD
/content/mamba/envs/pgd/bin/python test.py --ckpt pretrained/PGD.ckpt --resolutions 10000_poisson,50000_poisson --noise_lvls 0.01,0.02 --niters 2 --output_root /content/results/task2/PGD > /content/results_pgd_test.log 2>&1; echo "test.py exit $?"
date +%T
grep -B1 -A2 '^Mean' /content/results_pgd_test.log
cp -r /content/results $L335_DRIVE/ && cp /content/results_pgd_test.log $L335_DRIVE/results/task2/ && echo "copied to Drive"

22:56:41
test.py exit 0
23:15:05
[2026-11-14 22:58:42,981::PGD_test_PUNet_10000_poisson_0.01::INFO] 
Mean
cd_sph	1.797676995920
p2f	0.469398691166
--
[2026-11-14 23:05:56,405::PGD_test_PUNet_50000_poisson_0.01::INFO] 
Mean
cd_sph	0.476665095448
p2f	0.297362112178
--
[2026-11-14 23:07:50,260::PGD_test_PUNet_10000_poisson_0.02::INFO] 
Mean
cd_sph	2.430417938740
p2f	0.729434397726
--
[2026-11-14 23:15:04,717::PGD_test_PUNet_50000_poisson_0.02::INFO] 
Mean
cd_sph	0.615674607616
p2f	0.404785858336
copied to Drive


In [ ]:
%%bash
# Task 2, step 1: P2P-Bridge object model (PVDS_PUNet, xyz only) 

# (defaults: 10K and 50K points, 1/2/3% noise, --steps 5, --k 3, seed 42, no EMA); only the output folder is changed.
# The paper says its Table 1 used three denoising steps, so a second run uses --steps 3 (everything else the same).
source /content/p2pb_env.sh;
cd /content/P2P-Bridge
for s in 5 3; do date +%T; python evaluate_objects.py --model_path ./pretrained/PVDS_PUNet/latest.pth --dataset PUNet --steps $s --output_root /content/results/task2/P2P-Bridge > /content/results_p2pb_steps$s.log 2>&1; echo "steps $s exit $?"; done;
date +%T
for f in /content/results/task2/P2P-Bridge/PUNet/*/Summary_PUNet.csv; do echo "$f"; tail -1 "$f"; done;
cp -r /content/results $L335_DRIVE/ && cp /content/results_p2pb_steps*.log $L335_DRIVE/results/task2/ && echo "copied to Drive"

09:05:18
steps 5 exit 0
09:29:55
steps 3 exit 0
09:51:28
/content/results/P2P-Bridge/PUNet/P2P-Bridge_steps_3_10000_0.01/Summary_PUNet.csv
P2P-Bridge_steps_3,0.000226910439,0.000037911354
/content/results/P2P-Bridge/PUNet/P2P-Bridge_steps_3_10000_0.02/Summary_PUNet.csv
P2P-Bridge_steps_3,0.000322548233,0.000082676127
/content/results/P2P-Bridge/PUNet/P2P-Bridge_steps_3_10000_0.03/Summary_PUNet.csv
P2P-Bridge_steps_3,0.000402036485,0.000144453864
/content/results/P2P-Bridge/PUNet/P2P-Bridge_steps_3_50000_0.01/Summary_PUNet.csv
P2P-Bridge_steps_3,0.000058737591,0.000008644128
/content/results/P2P-Bridge/PUNet/P2P-Bridge_steps_3_50000_0.02/Summary_PUNet.csv
P2P-Bridge_steps_3,0.000091220102,0.000033758051
/content/results/P2P-Bridge/PUNet/P2P-Bridge_steps_3_50000_0.03/Summary_PUNet.csv
P2P-Bridge_steps_3,0.000155667352,0.000085076028
/content/results/P2P-Bridge/PUNet/P2P-Bridge_steps_5_10000_0.01/Summary_PUNet.csv
P2P-Bridge_steps_5,0.000228336586,0.000039257259
/content/results/P2P-Bridg

In [ ]:
%%bash
# Task 2, step 2a: noisy inputs at 2.5% (the public ScoreDenoise data only has 1, 2 and 3%)
# same code as the __main__ block of src/data/corruption.py in our repo (copied here because the repo is private): Gaussian noise with
# std = 2.5% of each shape's radius (bounding-box centre, farthest point), one generator with seed 2025, all 10K shapes
# first, then all 50K shapes, in alphabetical order; run once, it gives the same clouds as the group's run
cd /content
if test -d data/examples/PUNet_50000_poisson_0.025; then echo "2.5% inputs already there"; else
/content/mamba/envs/pgd/bin/python - <<'EOF'
import numpy; print("NumPy", numpy.__version__)
from pathlib import Path
import numpy as np

rng = np.random.default_rng(2025)

root = Path("data")
noise_level = 0.025

for resolution in ["10000_poisson", "50000_poisson"]:

    clean_dir = (root / "PUNet" / "pointclouds" / "test" / resolution)
    output_dir = (root / "examples" / f"PUNet_{resolution}_{noise_level}")

    # Preserve the previous experiment.
    if output_dir.exists():
        backup_dir = output_dir.with_name(output_dir.name + "_fixed_std_backup")

        if backup_dir.exists():
            raise FileExistsError(backup_dir)

        output_dir.rename(backup_dir)

    output_dir.mkdir(parents=True)

    for path in sorted(clean_dir.glob("*.xyz")):
        clean = np.loadtxt(path, dtype=np.float32)

        center = (clean.max(axis=0) + clean.min(axis=0)) / 2
        radius = np.linalg.norm(clean - center, axis=1).max()

        sigma = noise_level * radius

        noisy = clean + rng.normal(0.0, sigma, size=clean.shape).astype(np.float32)

        np.savetxt(output_dir / path.name, noisy, fmt="%.8f")

        print(f"{resolution:15s} {path.stem:15s} " f"radius={radius:.4f}, sigma={sigma:.5f}")

print("Generated radius-scaled 2.5% noisy clouds.")
EOF
fi
ls data/examples | grep 0.025
mkdir -p $L335_DRIVE/results/task2/inputs && cp -r data/examples/PUNet_*_0.025 $L335_DRIVE/results/task2/inputs/ && echo "copied to Drive"